# 03 — Causal Steering with the Persona Vector

**Goal:** Prove the vector is causal (not just correlational) by adding it to
the residual stream at decoding time and watching the trait appear.

Steering formula (paper Eq. 3.2):

$$h_\ell \leftarrow h_\ell + \alpha \cdot v_\ell$$

Positive α → induce the trait. Negative α → suppress it.

You'll:
1. Sweep α across `{0, 0.5, 1, 1.5, 2, 2.5}` and judge the responses.
2. Track **coherence** alongside trait expression — high α tends to break
   coherence, demonstrating the classic capability / safety tradeoff.


## Setup


In [ ]:
from persona_lib import ActivationModel, ChatPrompt, steering, load_trait, load_extraction
from persona_lib import score_trait, score_coherence
from persona_lib.viz import plot_alpha_sweep
import numpy as np
import matplotlib.pyplot as plt

TRAIT = 'evil'
model = ActivationModel("Qwen/Qwen2.5-7B-Instruct")
result = load_extraction(TRAIT)
L, v = result.best_layer, result.vectors_by_layer[result.best_layer]
artifacts = load_trait(TRAIT)
eval_prompts = [ChatPrompt(user=q) for q in artifacts.eval_questions[:8]]
print(f"steering at layer {L}, ||v|| = {np.linalg.norm(v):.2f}")


## TODO 1 — α sweep

For each α in the sweep, generate ONE response per eval prompt under
`steering(model, v, layer=L, alpha=α)`, then score trait expression AND
coherence. Store means.


In [ ]:
ALPHAS = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5]
trait_scores, coh_scores = [], []

for alpha in ALPHAS:
    # TODO: under `steering(...)`, generate one response per prompt, score both,
    # and append the means to the two lists.
    ...  # YOUR CODE HERE

print("α       trait  coherence")
for a, t, c in zip(ALPHAS, trait_scores, coh_scores):
    print(f"{a:>4.1f}   {t:>5.1f}   {c:>5.1f}")


## Plot the sweep


In [ ]:
plot_alpha_sweep(ALPHAS, trait_scores, coh_scores, title=f"Steering sweep — {TRAIT} (layer {L})")
plt.show()


## Qualitative: show steered generations side-by-side


In [ ]:
probe = ChatPrompt(user=artifacts.eval_questions[0])
print("=== α = 0 ===")
print(model.generate(probe, max_new_tokens=150, temperature=0.0)[0])

with steering(model, v, layer=L, alpha=2.0):
    print("\n=== α = 2.0 ===")
    print(model.generate(probe, max_new_tokens=150, temperature=0.0)[0])


## Analysis

1. At what α does trait score plateau? When does coherence start to collapse?
2. The paper (Fig. 7A) shows steering *against* the vector (α < 0) suppresses the
   trait but degrades MMLU at large |α|. Try `α = -2.0`: does the model actively
   refuse "evil" questions, or does it just get confused?
3. Pick one generation from the α=2.0 run and annotate it — what changed
   syntactically, semantically, or in framing?


## Capstone seed

- Try steering at multiple layers simultaneously via `steering_multi` (from
  `persona_lib.steering`). Does stacking help?
- Try steering *toward* one trait and *against* another (e.g., evil + anti-sycophancy).
